# Layer Normalization: From Theory to Implementation

This notebook implements **Layer Normalization** (Ba, Kiros, Hinton, 2016) from scratch using PyTorch. We will compare its performance against Batch Normalization and a baseline (no normalization) on the MNIST dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Set device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Dataset Preparation
We use the MNIST dataset. It is a standard proxy for examining convergence behavior in deep networks.

In [ ]:
# Configuration
BATCH_SIZE = 64

# Transformations: Normalize to aid convergence even for baseline
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load Data
train_set = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)

## 2. Custom Layer Normalization Implementation
Here we implement the core math of the paper. 

$$ y = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta $$

Crucially, $\mu$ and $\sigma$ are computed over the **last dimension** (features), independent of the batch size.

In [ ]:
class CustomLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super(CustomLayerNorm, self).__init__()
        self.normalized_shape = (normalized_shape,) if isinstance(normalized_shape, int) else normalized_shape
        self.eps = eps
        
        # Learnable parameters: Gamma (scale) and Beta (shift)
        self.gamma = nn.Parameter(torch.ones(self.normalized_shape))
        self.beta = nn.Parameter(torch.zeros(self.normalized_shape))

    def forward(self, x):
        # 1. Calculate Mean over the feature dimension (dim=-1)
        mean = x.mean(dim=-1, keepdim=True)
        
        # 2. Calculate Variance over the feature dimension
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        
        # 3. Normalize
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        
        # 4. Scale and Shift
        output = x_norm * self.gamma + self.beta
        return output

## 3. Model Definition
We define a generic Deep Feed-Forward Network that accepts a normalization type argument (`none`, `batch`, `layer`).

In [ ]:
class DeepFeedForward(nn.Module):
    def __init__(self, norm_type="none"):
        super(DeepFeedForward, self).__init__()
        self.flatten = nn.Flatten()
        
        layers = []
        
        # Hidden Layer 1
        layers.append(nn.Linear(28*28, 512))
        if norm_type == "batch":
            layers.append(nn.BatchNorm1d(512))
        elif norm_type == "layer":
            layers.append(CustomLayerNorm(512))
        layers.append(nn.ReLU())
        
        # Hidden Layer 2
        layers.append(nn.Linear(512, 512))
        if norm_type == "batch":
            layers.append(nn.BatchNorm1d(512))
        elif norm_type == "layer":
            layers.append(CustomLayerNorm(512))
        layers.append(nn.ReLU())
        
        # Output Layer
        layers.append(nn.Linear(512, 10))
        
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

## 4. Training Loop
We train 3 variants for 5 epochs to observe initial convergence speed.

In [ ]:
def train_model(norm_type, epochs=5, lr=0.001):
    print(f"Starting training for: {norm_type.upper()}")
    model = DeepFeedForward(norm_type=norm_type).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    loss_history = []
    acc_history = []
    
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        
        for i, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            # Log loss every 50 batches
            if i % 50 == 0:
                loss_history.append(loss.item())
        
        epoch_acc = 100 * correct / total
        acc_history.append(epoch_acc)
        print(f"Epoch [{epoch+1}/{epochs}] Acc: {epoch_acc:.2f}%")
        
    return loss_history, acc_history

In [ ]:
# Run Experiments
loss_none, acc_none = train_model("none")
loss_batch, acc_batch = train_model("batch")
loss_layer, acc_layer = train_model("layer")

## 5. Visualization
Comparing Loss Convergence and Validation Accuracy.

In [ ]:
def smooth(scalars, weight=0.9):
    last = scalars[0]
    smoothed = []
    for point in scalars:
        smoothed_val = last * weight + (1 - weight) * point
        smoothed.append(smoothed_val)
        last = smoothed_val
    return smoothed

plt.figure(figsize=(15, 6))

# Plot 1: Loss
plt.subplot(1, 2, 1)
plt.plot(smooth(loss_none), label='No Norm', alpha=0.6)
plt.plot(smooth(loss_batch), label='Batch Norm', alpha=0.6)
plt.plot(smooth(loss_layer), label='Layer Norm', linewidth=2, color='green')
plt.xlabel('Training Steps (x50)')
plt.ylabel('Cross Entropy Loss')
plt.title('Training Loss Convergence')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Accuracy
plt.subplot(1, 2, 2)
plt.plot(acc_none, marker='o', label='No Norm')
plt.plot(acc_batch, marker='s', label='Batch Norm')
plt.plot(acc_layer, marker='^', label='Layer Norm')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.title('Validation Accuracy vs Epochs')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Interpretation:**
- **Loss Curve:** You should observe that LayerNorm (green) drops significantly faster than the 'No Norm' baseline, demonstrating the acceleration of training.
- **Accuracy:** Similarly, LayerNorm achieves high accuracy in the very first epoch compared to the baseline. While BatchNorm also performs well here, LayerNorm offers these benefits independent of batch size, which is critical for other architectures like RNNs.